# The Impact of M&A on Inventor Mobility and Innovation
## Draft notebook for the GitHub package

This notebook is a narrative companion to the construction and analysis pipeline in the repository. It is written to do four jobs at once:

1. explain how the data are built and how the main empirical designs work,
2. present a disciplined headline story,
3. distinguish robust findings from fragile or method-dependent ones,
4. serve as a base for a sequence of public-facing writeups such as LinkedIn posts.

## Current headline story

The cleanest story is **not** that mergers and acquisitions uniformly reduce innovation output.  
The stronger and more defensible claim is narrower:

> **M&A appears to change inventor mobility and the direction of inventive search more clearly than it changes aggregate output, with especially important effects on the acquiror side.**

In particular, the most interesting pattern is that **acquiror-side inventor mobility rises after deals, and exploration becomes more prominent**. By contrast, effects on patents, citations, and related output measures are often more fragile across methods or complicated by pre-trends.

## What this notebook treats as primary versus secondary

### Primary findings
- Acquiror-side post-deal inventor mobility.
- Acquiror-side exploration outcomes.
- Heterogeneity by firm size, especially where negative baseline effects weaken among larger firms.

### Secondary findings
- Aggregate output measures such as patent counts, citations, and quality proxies.
- Arrivals and departures as supporting evidence for organizational reshuffling.
- Target-side effects, unless they are unusually stable across methods.

## Important caution

This draft reflects the current state of the project and the review notes. Some statements below are intentionally cautious because a few results are sensitive to estimator choice, event-study shape, or pre-trend concerns.

## Repository orientation

A clean GitHub version of the project should separate the work into:

- **construction scripts**: build cleaned firm-year and inventor-year / inventor-event panels,
- **analysis scripts**: baseline DiD, event studies, heterogeneity, and advanced estimators,
- **output folders**: regression tables, coefficient plots, and summary CSVs,
- **notebooks**: one main narrative notebook, plus optional shorter diagnostics notebooks.

This notebook is meant to be the **main interpretive notebook** rather than the place where every heavy computation happens.


## Construction of the dataset: overview and design logic

A major part of the value of this project is the construction itself. The empirical design is only credible if the data pipeline is transparent about how raw patent, accounting, and M&A records are converted into analysis-ready panels.

This section therefore documents the construction in the same spirit as a methods section in an academic paper, but with a stronger emphasis on engineering choices, reproducibility, and economic interpretation. The target reader is a technically literate economist, data scientist, or recruiter who wants to understand not only *what* was estimated, but *how* the underlying research asset was built.

### What the construction is trying to achieve

The pipeline solves four linked problems:

1. **Link inventions to inventors and firms.**  
   Patent records are inventor-centric and assignee-centric, while the firm-level analysis requires a stable public-firm identifier.

2. **Build meaningful innovation measures rather than just patent counts.**  
   The project adds citation-based quality proxies, novelty measures, exploration/exploitation metrics, and technology proximity scores.

3. **Identify economically meaningful inventor moves.**  
   The mobility analysis is not based on one-off assignee changes. It uses a stricter move definition designed to reduce false positives from noisy patent assignment histories.

4. **Create panels aligned to M&A timing.**  
   The end product is not merely a collection of cleaned files. It is a set of firm-year, inventor-year, and inventor-event panels centered on merger timing and suitable for DiD and event-study analysis.

### Why a multi-stage pipeline is necessary

No single source contains the final object needed for the project.

- **PatentsView** provides patent, inventor, assignee, location, CPC, and citation information.
- **External patent-quality data** add KPSS-based measures such as `xi_real` and citation fields.
- **Compustat** provides firm fundamentals.
- **WRDS link tables** connect Compustat firms to CRSP/market identifiers such as `permco`.
- **SDC M&A data** provide acquisition and target events.

The pipeline therefore moves from raw source-specific files to increasingly integrated research objects:

> raw patent and firm data → enriched patent-inventor-firm file → mobility and technology measures → final firm-year and inventor-event analysis panels

### Construction philosophy

Three choices are worth stating up front.

**First, the pipeline is intentionally conservative about identifiers.**  
The core analytical firm identifier is `permco`, because it gives a stable public-firm identifier that can be used across patents, market data, and M&A events.

**Second, intermediate files are cached aggressively.**  
This is partly a speed issue and partly a reproducibility issue. Some steps are expensive enough that a serious empirical workflow benefits from explicit checkpoints.

**Third, the split repository structure preserves the logic of the original monolithic construction script.**  
The runner executes modules in sequence using a shared namespace. That is not the most object-oriented design possible, but it is a sensible choice for a research codebase where preserving validated logic is often more important than abstract elegance.



## Construction pipeline at a glance

The construction code is organized into eight sequential modules plus one runner script.

| File | Main purpose | Main outputs |
|---|---|---|
| `run_construction.py` | Executes all construction modules in order | End-to-end pipeline |
| `01_setup_helpers.py` | Paths, imports, global helpers, data download helper | Shared environment |
| `02_patent_panel_construction.py` | Builds the core patent-inventor-firm dataset and patent-level measures | `pat_inv_firm_df.pkl` and intermediate patent objects |
| `03_exploration_exploitation.py` | Adds exploration/exploitation metrics | enriched patent-inventor-firm file |
| `04_mobility_and_mover_metrics.py` | Detects inventor moves and builds move-related performance objects | mover event and move-performance files |
| `05_technology_similarity.py` | Computes technology proximity and rolling similarity measures | event-based and rolling similarity outputs |
| `06_firm_fundamentals.py` | Builds Compustat-based firm fundamentals and links them to `permco` | linked Compustat panel |
| `07_linking_and_manda.py` | Cleans and links M&A deals and adds pre-deal technology similarity | `manda.pkl` |
| `08_final_panels.py` | Produces the final firm-year, firm-event, inventor-year, and inventor-event panels | final analysis panels |

### Pipeline orchestration

The runner preserves the original notebook-like dependency structure:

```python
EXECUTION_ORDER = [
    "01_setup_helpers.py",
    "02_patent_panel_construction.py",
    "03_exploration_exploitation.py",
    "04_mobility_and_mover_metrics.py",
    "05_technology_similarity.py",
    "06_firm_fundamentals.py",
    "07_linking_and_manda.py",
    "08_final_panels.py",
]
```

and then executes the files in a **shared global namespace**:

```python
shared_globals = runpy.run_path(
    str(script_path),
    init_globals=shared_globals,
    run_name="__main__",
)
```

This is a deliberate research-engineering compromise. It keeps the split files readable and GitHub-friendly while minimizing the risk of introducing logic changes during refactoring.



## Step 0. Environment, paths, and helper infrastructure

The setup module does four jobs:

1. loads the core Python stack,
2. defines the project paths,
3. validates that required local inputs exist,
4. provides a helper to download PatentsView files on demand.

### Main libraries used

This construction is fundamentally a **Python data engineering and empirical research pipeline**. The main tools are:

- **pandas** and **NumPy** for table manipulation and vectorized operations,
- **scikit-learn** for matching support and nearest-neighbor logic,
- **tqdm** for progress monitoring in long-running loops,
- **requests** and **zipfile** for downloading and unpacking PatentsView files,
- **collections.Counter / defaultdict / deque** for efficient rolling technology-history objects.

The setup code also defines a strict path check:

```python
def assert_required_paths_exist():
    required_paths = [
        BASE_PROJECT_PATH,
        RAW_DATA_PATH,
        FINANCIAL_DATA_PATH,
        MANDA_DATA_PATH,
        LINKTABLE_CSV,
    ]
    for path in required_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required path does not exist: {path}")
```

This matters because the project depends on several local raw-data directories. Failing early is better than discovering a missing file after several hours of preprocessing.

### On-demand raw-data loading

A useful design choice is the helper that downloads PatentsView files only if they are not already stored locally:

```python
def download_and_load_patentsview_data(file_name, **kwargs):
    base_url = 'https://s3.amazonaws.com/data.patentsview.org/download'
    local_file_path = os.path.join(RAW_DATA_PATH, file_name)
    if os.path.exists(local_file_path):
        print(f"Loading '{file_name}' from local directory.")
    else:
        r = requests.get(f"{base_url}/{file_name}.zip", timeout=300)
        r.raise_for_status()
        with zipfile.ZipFile(BytesIO(r.content)) as z:
            z.extractall(RAW_DATA_PATH)
    return pd.read_csv(local_file_path, delimiter="\t", ...)
```

Economically, this step is mundane. Methodologically, it is important. It makes the project reproducible from raw public patent files instead of relying on opaque manual extracts.

### Why this matters for the notebook narrative

For an employer-facing research notebook, this section demonstrates that the project is not just a set of regressions. It is a full empirical data asset with explicit input validation, caching, and recoverability.



## Step 1. Build the core patent-inventor-firm dataset

This is the foundational construction step. Everything later in the project depends on producing a clean patent-level file that links:

- a patent,
- its inventor(s),
- the public firm to which it can be assigned,
- the patent's technological and quality characteristics.

### Raw patent inputs

The construction draws the following core tables from PatentsView:

- inventor-patent links,
- assignee-patent links,
- inventor locations,
- CPC technology classifications,
- patent-to-patent citations,
- patent issue dates,
- application filing dates.

The code begins with a cache-first wrapper:

```python
def build_or_load_pat_inv_firm(cache_path=None, rebuild=False):
    if (not rebuild) and os.path.exists(cache_path):
        return pd.read_pickle(cache_path)
```

This is good empirical practice because the patent core is both expensive and stable. Once validated, it should not be rebuilt unless upstream logic changes.

### Cleaning and identifier discipline

Several cleaning choices are substantive rather than cosmetic.

#### 1. Keep business assignees and the primary assignee
The assignee file is filtered to:

```python
assignee_df = assignee_df[
    (assignee_df['assignee_type'] == 2) &
    (assignee_df['assignee_sequence'] == 0)
].copy()
```

This means the construction is intentionally focused on the primary business assignee rather than all possible assignee relationships. That is a defensible choice because the downstream goal is to connect patents to publicly listed firms in a way that is stable enough for event-study analysis.

#### 2. Use filing year rather than issue year for many research objects
The code merges issue dates and application dates and defines:

```python
patent_df['filing_year'] = patent_df['filing_date'].dt.year
```

This is economically sensible because filing dates are closer to the timing of inventive activity than grant dates, which can be delayed by examination and administrative processes.

#### 3. Retain detailed CPC information
The code keeps CPC subclasses and groups, using primary CPC assignments for some tasks and full CPC combinations for novelty construction later. This matters because technology direction is central to the project's mechanism story.

### External enrichments

Two external merges are especially important.

#### KPSS patent-quality data
The construction merges patent-level `xi_real` and citation information from an external replication dataset and links patents to `permco`. This is how the project transitions from patent objects to firm-identified patent objects.

#### State-level noncompete enforceability
The code merges a state-year noncompete enforcement score using inventor state and filing year:

```python
pat_inv_firm_df = pat_inv_firm_df.merge(
    nca_df[['filing_year', 'state_fips', 'std_score']],
    on=['filing_year', 'state_fips'], how='left'
).rename(columns={'std_score': 'nca_enforce_score'})
```

This is a useful example of economically informed enrichment. The project is about inventor mobility, so including a local legal environment relevant to mobility is conceptually well motivated.

### Patent-level quality measures

The construction does not rely on one patent-quality measure. It builds several.

#### Forward citations
First, unconditional forward citations are computed from citation links. Then the code constructs class-year normalized bins based on CPC subclass and filing year:

```python
stats = df.groupby(['filing_year', 'cpc_subclass'])['forward_citations']           .quantile([0.90, 0.99]).unstack()
```

This is good measurement design. Raw citations are noisy across technology classes and cohorts. Ranking patents within technology-year cells makes the quality proxy more comparable.

#### KPSS-based citation bins
The same binning logic is repeated for the KPSS citation field. This redundancy is useful: it reduces dependence on a single citation definition.

### Patent novelty

The novelty measure is based on new combinations of CPC classes, following the logic of recombinatorial innovation. The key idea is simple:

- represent a patent as a set of technology classes,
- enumerate all within-patent class pairs,
- identify whether each pair is new in the historical record,
- summarize the share of new combinations.

A simplified version of the core logic is:

```python
def _canonical_pair(a, b):
    return f'{a}_{b}' if a <= b else f'{b}_{a}'

pairs = g.assign(pair_key=lambda df: df['cpc_tuple'].apply(_make_pairs)).explode('pair_key')
first_date_per_pair = pairs.groupby('pair_key')['issue_date_dt'].min()
```

This is an intellectually strong part of the construction because it goes beyond volume and average quality. It directly measures whether the patent recombines knowledge in a way that appears novel relative to prior art.

### Citation-link measures

The pipeline also builds:

- backward citations,
- self-citations at the firm level.

The self-citation routine is written carefully to be memory-safe by operating in chunks and comparing assignee sets through set intersections. That is a strong engineering choice for large citation graphs.

### Final patent-inventor-firm object

After all merges, the project creates the core research object:

```python
pat_inv_firm_df = pat_inv_df.merge(patent_df, on='patent_id', how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(kpss_df, on=['patent_id'], how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(patent_novelty_df, on='patent_id', how='left')
...
```

It then adds inventor career features such as:

- first filing year,
- first CPC field,
- inventor age measured as years since first filing,
- indicators for whether a patent stays close to the inventor's original field.

### Why this step is done this way

This construction step is designed to create a **single enriched micro-level innovation file** from which multiple downstream panels can be derived without rebuilding patent logic each time. That is both computationally efficient and methodologically cleaner.



## Step 2. Construct exploration and exploitation measures

One of the most interesting aspects of the project is that it tries to distinguish **direction of search** from **quantity of output**.

The key intuition is that mergers may reshape what inventors and firms work on even when the total number of patents does not move much.

### Measure design

For each patent, the code builds a set of CPC subclasses and compares that set to the recent knowledge base of:

- the patent's inventor team,
- the patent's firm or firms.

The knowledge base is defined using a rolling five-year window.

A simplified version of the logic is:

```python
for row in patent_level.itertuples(index=False):
    inv_knowledge = union_of_recent_cpcs_for_inventors
    firm_knowledge = union_of_recent_cpcs_for_firms

    exp_inv = 1 - overlap(current_cpcs, inv_knowledge) / len(current_cpcs)
    exp_firm = 1 - overlap(current_cpcs, firm_knowledge) / len(current_cpcs)
```

### Why this is economically meaningful

A patent is classified as more exploratory when it uses CPC classes that are less represented in the inventor's or firm's recent history. This is an intuitive way to measure movement into less familiar technological space.

This is also one reason the project's headline story is more persuasive for exploration than for aggregate patent counts. Exploration is closer to the organizational mechanism one might expect after an acquisition:

- new teams are combined,
- internal allocation changes,
- inventors may be redeployed,
- firms may search more broadly or in adjacent spaces.

### Why the implementation uses rolling deques

The implementation stores prior technology histories in `deque` objects and purges outdated years as it moves through the patent sequence. That makes the rolling-window computation efficient enough to scale while remaining transparent.



## Step 3. Identify inventor mobility events and build mover metrics

A central contribution of the project is that inventor mobility is not treated as a noisy byproduct. It is directly measured and then connected to post-M&A firm outcomes.

### Strict move identification

The move definition is intentionally conservative. The code first restricts attention to inventors with at least five patents:

```python
min_patents = 5
prolific_inventors = inventor_counts[inventor_counts >= min_patents].index
```

Then it defines a move only when there is a stable firm sequence around the transition:

- two patents before the move at the old firm,
- the first patent at the new firm,
- two subsequent patents at the new firm.

The core rule is:

```python
is_move = (
    (career_df['permco'] != career_df['prev_permco']) &
    (career_df['prev2_permco'] == career_df['prev_permco']) &
    (career_df['next_permco'] == career_df['permco']) &
    (career_df['next2_permco'] == career_df['next_permco'])
)
```

### Why this strict rule is useful

Patent assignment data can be noisy. A single assignee switch may reflect legal assignment timing, joint work, or temporary data noise rather than a real labor-market transition.

The stricter move rule sacrifices some sample size, but it likely improves signal quality, which is exactly the right tradeoff for a project whose main mechanism involves labor reallocation.

### Pre/post mover performance

Once moves are defined, the code builds five-year pre-move and post-move performance windows and aggregates inventor outcomes such as:

- patent counts,
- citations,
- `xi_real`,
- novelty,
- backward and self-citations,
- exploration and exploitation,
- team size.

This produces both:
- a long panel with one row per inventor-move-period, and
- a wide format that makes change calculations straightforward.

### Benchmarking movers relative to peers

The project also compares movers to peers at origin and destination firms. That is a subtle but valuable addition because it helps distinguish:

- whether firms are losing unusually important inventors,
- whether incoming inventors look stronger or weaker than incumbent peers,
- whether post-move performance changes suggest integration frictions or gains.

From a research-design perspective, this is a strong bridge between micro labor allocation and firm-level organizational outcomes.



## Step 4. Measure technology similarity around moves and deals

The project next constructs technology-similarity objects that help interpret whether mobility and M&A connect technologically related or unrelated domains.

### Event-based proximity around inventor moves

The first routine compares technology vectors before and after a move for:

- the inventor,
- the origin firm,
- the destination firm.

Vectors are represented as `Counter` objects over CPC subclasses, and similarity is measured with cosine similarity:

```python
def _calculate_cosine_similarity(vec1, vec2):
    dot_product = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in all_keys)
    norm1 = sqrt(sum(v**2 for v in vec1.values()))
    norm2 = sqrt(sum(v**2 for v in vec2.values()))
    return dot_product / (norm1 * norm2)
```

This yields quantities such as:

- inventor pre/post self-similarity,
- inventor similarity to origin firm,
- inventor similarity to destination firm,
- origin-destination firm similarity.

### Rolling annual similarity

The second routine creates a rolling annual similarity measure, comparing a current year's technology vector with the preceding five-year technology history. This is helpful because it turns technology direction into a panel variable rather than a one-time event summary.

### Why these measures matter

These objects are not the headline outcomes of the project, but they greatly improve interpretation. If mobility rises after deals, the next question is whether it reflects:

- redeployment into familiar technologies,
- integration into the acquiring firm's technological base,
- broader exploratory search.

Technology similarity measures help answer that question.



## Step 5. Build firm fundamentals from Compustat and link them to market identifiers

The firm-side empirical analysis needs more than patent outcomes. It needs standard controls and heterogeneity variables describing firm size, profitability, leverage, liquidity, investment, and financial constraints.

### Sample construction in Compustat

The code filters Compustat to a standard industrial sample:

- excludes financials, utilities, and public sector entities,
- keeps industrial, standardized, consolidated, domestic statements,
- removes clearly invalid negative values for core accounting variables.

It then defines the analysis year as:

- same calendar year if the fiscal year ends in June to December,
- previous calendar year otherwise.

That timing rule is sensible because it aligns fiscal records more closely with the year to which they economically belong.

### Feature engineering

The module then constructs a broad set of variables, including:

- size: `log_at`, `log_sale`, `log_mv`,
- profitability: `roa`, margins, operating profitability,
- growth and valuation: `sale_growth`, `tobinsq`, `market_to_book`,
- financing and liquidity: leverage, cash, interest coverage,
- investment and composition: capital expenditures, R&D intensity, intangible assets,
- payout and repurchases,
- constraint indices such as **Whited-Wu** and **Hadlock-Pierce**.

This is more than a cosmetic merge. It creates the economic state variables needed to:
- control for confounding firm conditions,
- define heterogeneity splits,
- interpret M&A effects in a richer organizational context.

### Real scaling and CPI adjustment

The code downloads CPI data from FRED and uses it to build real assets and real R&D variables. That is another sign that the construction is meant to support serious empirical work rather than only descriptive charts.

### Linking Compustat to `permco`

The link to the patent side is done using the CRSP-Compustat link table. The code keeps primary, validated links and restricts them to valid date ranges.

```python
compustat_core_linked = compustat_core_df.merge(
    linktable[['gvkey', 'permco', 'linkdt', 'linkenddt']],
    on='gvkey', how='inner'
)
compustat_core_linked = compustat_core_linked[
    (compustat_core_linked['datadate_dt'] >= compustat_core_linked['linkdt']) &
    (compustat_core_linked['datadate_dt'] <= compustat_core_linked['linkenddt'])
]
```

This link discipline is essential. The project would be much weaker if the public-firm mapping were fuzzy or time-inconsistent.



## Step 6. Clean and link M&A deals

The M&A module transforms raw SDC transaction records into a deal panel that can be merged into firm-year and inventor-year data.

### Deal cleaning choices

Several choices are worth highlighting.

#### 1. Announcement timing is central
The project uses announcement-year timing for the event-study setup. That is reasonable because announcement is typically the point at which firms, investors, and employees begin responding to the transaction.

#### 2. Link both target and acquiror through CUSIP-6 and valid date windows
The code truncates CUSIPs to six digits and merges both sides of the deal through the link table. It then keeps only observations for which the announcement date falls within the valid link ranges for both firms.

#### 3. Restrict deal outcomes
The code keeps completed and withdrawn deals and explicitly creates a failed-merger indicator. This is useful because failed deals can later be used either as controls, robustness checks, or descriptive contrasts.

### Deal-level variables

The M&A panel retains:

- acquiror and target identifiers,
- transaction value,
- ownership percentages,
- diversifying indicators based on industry codes,
- deal outcome and failure status.

### Pre-deal technology similarity between target and acquiror

A particularly strong addition is the pre-deal technology similarity measure, built from five-year patent portfolios of the target and acquiror.

This is useful for at least two reasons:

1. it helps characterize whether deals combine related or more distant technology portfolios,
2. it provides a natural heterogeneity variable for later interpretation.

In other words, the M&A panel is not just a timing file. It is a rich deal-characterization file.



## Step 7. Assemble the final analysis panels

The final module turns the intermediate construction objects into the panels used for estimation.

This is the point where the project shifts from **data integration** to **econometric design**.

### 7.1 Firm-year panel

The code first aggregates patent outcomes to the `permco × year` level:

```python
firm_year_patent_metrics = pat_inv_firm_df.groupby(['permco', 'filing_year']).agg(
    total_patents=('patent_id', 'nunique'),
    cites=('cites', 'sum'),
    xi_real=('xi_real', 'sum'),
    ...
).reset_index()
```

It then merges in:

- rolling firm technology similarity,
- Compustat fundamentals,
- inventor arrival and departure measures,
- relative quality of arriving and departing inventors,
- M&A event counts and deal-value measures.

The result is a firm-year panel that simultaneously describes:
- innovative output,
- technology direction,
- labor flows,
- financial conditions,
- M&A exposure.

That integration is one of the most compelling aspects of the project.

### 7.2 Firm-year M&A event-study panel

The next object is a firm-year event-study panel centered on the **closest deal year** within a ±5-year window.

A nice engineering feature here is that the code avoids `merge_asof` and instead computes previous and next deal indices explicitly with `np.searchsorted`. That makes the logic more transparent and less fragile.

The panel then defines:

- `ma_deal_role` = acquiror, target, or no recent M&A,
- `years_from_ma_deal`,
- deal value,
- failed-merger flag,
- other-party identifier,
- pre-deal technology similarity.

This object is the main firm-level treatment panel used for DiD and event-study work.

### 7.3 Inventor-year panel

The inventor-year panel is built for inventors who ever patent at firms in the analysis universe. The code creates:

- annual inventor outcome measures,
- zero-filled years for no-patenting observations within the inventor career span,
- annual employer assignment,
- move-year indicators,
- M&A context of the assigned employer.

This is an ambitious and important design choice. It converts sparse patent observations into a panel suitable for longitudinal analysis of inventor behavior.

### 7.4 Inventor-event panel with matched controls

The most sophisticated construction step is the inventor M&A event-study panel.

The logic is:

1. identify treated inventor-deal events,
2. keep only events feasible over the full event window,
3. construct matched control pseudo-events using inventor characteristics at `t = -1`,
4. expand each event into a balanced `[-5, +5]` relative-year grid,
5. merge annual inventor outcomes back onto the event grid.

The matching features include variables such as:

- inventor age,
- cumulative patents,
- cumulative citations,
- first CPC subclass.

The code supports propensity-score-based matching with exact matching on selected categorical fields.

This is a strong design because it tries to build a credible inventor-level counterfactual while keeping the event-study object balanced and interpretable.

### Why these final panels are the right endpoint

By the end of the pipeline, the project has produced four conceptually distinct but connected research objects:

1. **firm-year panel** for aggregate organizational outcomes,
2. **firm event-study panel** for treatment-timing analysis,
3. **inventor-year panel** for individual-level longitudinal behavior,
4. **inventor-event panel** for matched event-study analysis.

That architecture reflects a PhD-level empirical strategy: it links micro mechanism evidence to firm-level outcomes rather than relying on only one level of aggregation.



## Construction choices that are especially strong, and a few places to be explicit about limitations

### What is especially strong in the construction

1. **The project builds real research infrastructure, not just a regression-ready CSV.**  
   The layered construction from raw sources to multiple final panels is one of the clearest signs of technical maturity in the repository.

2. **The measurement choices are economically motivated.**  
   Novelty, exploration, mobility, and technology similarity are not generic machine-learning features. They are tailored to the economic mechanisms of the question.

3. **The pipeline balances ambition with tractability.**  
   Intermediate caching, chunked citation routines, and explicit path validation show a practical understanding of what large empirical projects require.

4. **The final objects are designed for multiple estimators.**  
   The construction is clearly shaped by downstream DiD, event-study, and matching requirements.

### A few limitations or caveats worth stating openly

A strong notebook should also be explicit about where the construction is deliberately imperfect.

- **Public-firm scope.**  
  The use of `permco` is analytically powerful, but it means the project is strongest for the publicly listed-firm universe rather than the full economy of patent assignees.

- **Assignee-based employer inference.**  
  Employer assignment from patent assignees is sensible and standard in this setting, but it is still an inferred labor-market link rather than a direct HR record.

- **Strict move definitions trade off recall for precision.**  
  That is a feature, not a bug, but it should be stated clearly.

- **The split modules still rely on sequential shared-state execution.**  
  For a research repository, this is acceptable and often desirable for faithfulness. For a production software package, one would likely move further toward explicit function imports and typed interfaces.

### One issue I would state honestly

The setup file still contains a comment suggesting that some foundational economic-data loading is "assumed to be here for brevity." In the current split repository, those later steps are handled by subsequent modules. This does not appear to break the pipeline, but the notebook should describe the actual implemented sequence rather than the older monolithic-script commentary.

That is an example of the kind of small inconsistency worth acknowledging rather than hiding.



## Summary of the construction

The construction pipeline creates a research dataset that is substantially richer than a standard event-study input file.

### In one sentence

The project starts from raw patent, accounting, and deal records and builds a linked set of firm-level and inventor-level panels that can speak to both **organizational outcomes** and **microeconomic mechanisms** around M&A.

### What each stage contributes

- **Setup and orchestration** establish a reproducible research environment.
- **Patent panel construction** creates the core patent-inventor-firm microdata.
- **Quality and novelty measures** turn patents into economically meaningful innovation objects.
- **Exploration and exploitation metrics** capture the direction of technological search.
- **Mobility construction** identifies meaningful inventor moves and summarizes mover quality.
- **Technology similarity measures** help interpret whether movement reflects related or distant technological reallocation.
- **Firm fundamentals and link tables** anchor the patent side in standard corporate-finance measurement.
- **M&A linking** creates timed treatment events with deal characteristics.
- **Final panels** turn all of the above into objects suitable for DiD, event studies, heterogeneity analysis, and matched inventor-event designs.

### Why this matters for the broader project

For recruiters and economists at technology firms, the construction is important because it demonstrates a combination of skills that often appear separately but less often together:

- research design,
- large-scale data construction,
- identifier linking across sources,
- thoughtful feature engineering,
- sensitivity to measurement error,
- and an ability to structure the final output around a clear economic question.

That combination is a major part of what makes the project a credible showcase of independent empirical research skill.


## Analysis workflow: overview and design logic

The analysis side of the project is organized much more explicitly than the construction side. The repository-facing analysis code does **not** recreate the original monolithic script by sharing a live namespace across modules. Instead, it uses a cleaner split between configuration, data loading, reusable estimator functions, and two high-level runners:

- a **firm-panel branch**, and
- an **inventor-year / inventor-event branch**.

That choice is economically and computationally sensible. The firm-level and inventor-level designs answer related but distinct questions, use different units of observation, and have different memory bottlenecks. Keeping them separate makes the workflow easier to inspect and easier to explain to external readers.

At a high level, the logic is:

1. load the final panels produced by construction,
2. define the treatment and event-time structure,
3. run baseline two-way fixed-effects DiD and event studies,
4. layer on heterogeneity specifications,
5. run selected staggered-adoption robust estimators,
6. use placebo and balance diagnostics to decide what is credible,
7. interpret only the results that survive those checks.

That last step is important. The notebook should not present the analysis as “run many methods until something is significant.” It should present the analysis as **triangulation**: multiple estimators are used because the treatment is staggered, treatment effects can be heterogeneous, and simple TWFE event studies can be misleading.


## Analysis pipeline at a glance

The public workflow is intentionally split into small files that do one thing each.

### Core orchestration
- `run_analysis.py` calls the two main branches in sequence.
- `run_firm_panel_analysis.py` contains the full firm-level workflow.
- `run_inventor_year_analysis.py` contains the inventor-year workflow.

### Shared infrastructure
- `config.py` defines paths, windows, and global switches.
- `data.py` loads the cleaned panels and merges lagged firm controls into the inventor-level datasets.
- `utils.py` contains the reusable econometric helpers such as the two-way fixed-effects `PanelOLS` wrapper and the heterogeneity `Z` constructors.

### Topical method modules
- `firm_analysis.py` contains the baseline firm DiD/event-study workflow, the matched stacked design, and the generic triple-DiD/event-study machinery.
- `inventor_year.py` contains inventor-event preparation, within-firm heterogeneity splits, downsampling for advanced methods, and the inventor-year CSDID wrapper.
- `advanced_methods.py` contains the advanced estimators used as cross-checks: Causal Forest, Synthetic Control, Sun–Abraham, and Borusyak–Jaravel–Spiess.
- `placebos.py` contains lead and permuted placebo timing exercises.
- `summary_statistics.py` writes compact panel summaries and pre-period baselines.

This split is a strength of the repository version. It makes the analysis look like a serious research pipeline rather than a notebook that happened to work once.


## Step A. Configuration and loading of the analysis data

The analysis starts with a compact configuration object:

```python
@dataclass
class AnalysisConfig:
    analysis_window: tuple[int, int] = (1980, 2020)
    event_study_window: tuple[int, int] = (-5, 5)
    event_study_ref_k: int = -1
    run_inventor_year_advanced: bool = True
    advanced_alpha: float = 0.05
    inventor_year_csdid_bootstrap_draws: int = 30
```

This is good practice for a public research project. It puts the key design choices in one place:

- the **calendar window** for the usable panel,
- the **event-study window**,
- the omitted event-time reference period,
- and the switches controlling the more expensive advanced methods.

The data loader then reads the final construction outputs and attaches lagged firm controls to the inventor-side panels. That is a subtle but important design step. The inventor outcomes are naturally inventor-level, but the mechanism may still run through the characteristics of the employing firm. By merging lagged firm controls into the inventor panels, the project can ask whether inventor-level changes persist after accounting for the employer's pre-period scale and financial condition.

Conceptually, this means the analysis combines:
- **individual outcomes**,
- **firm event timing**, and
- **firm baseline controls**.

That is exactly the kind of multi-level empirical setup that signals strong applied micro / innovation-economics training.


## Step B. Firm-panel design: what the firm analysis is trying to estimate

The firm branch asks a relatively direct question:

> When a publicly listed firm becomes an acquiror or a target, how do its innovation and inventor-flow outcomes evolve relative to otherwise similar firms without a recent M&A event?

The raw event panel is first turned into a standard DiD/event-study input:

```python
did_df["Treated"] = (did_df["ma_deal_role"] != "no_recent_MandA").astype(int)
did_df["Post"] = (did_df["years_from_ma_deal"].fillna(-1) >= 0).astype(int)
did_df["Post_Treated"] = did_df["Treated"] * did_df["Post"]
did_df["gname"] = np.where(
    did_df["years_from_ma_deal"].isna(),
    0,
    did_df["data_year"] - did_df["years_from_ma_deal"],
).astype(int)
```

Economically, these objects do the following:

- `Treated` identifies firms exposed to an M&A event.
- `Post` marks the post-event period.
- `Post_Treated` is the standard average post-treatment effect regressor.
- `gname` stores the cohort year, which is essential for staggered-treatment methods.

The code then creates separate datasets for:
- **acquiror vs. no recent M&A**, and
- **target vs. no recent M&A**.

That separation is a very good design choice. Acquirors and targets should not be pooled mechanically. The organizational mechanisms are different:
- acquirors face integration, reallocation, and coordination decisions;
- targets face absorption, disruption, or reorganization from the receiving side.

Treating them as separate empirical objects gives the analysis a cleaner interpretation.


## Step C. Why the firm analysis uses a matched stacked design

The baseline firm analysis does **not** just run one pooled two-way fixed-effects regression on all firms. It first builds a matched stacked panel.

The matching function works cohort by cohort. At the year before treatment, it uses:
- industry (`sic3`),
- firm size (`log_sale`, `log_mv`),
- and a cohort-specific propensity score,

to pair treated firms with comparable controls. Within industry, the actual pair selection uses Mahalanobis distance on firm size variables, with an optional propensity-score caliper.

A compact summary of the matching logic is:

```python
for cohort_year in cohort_years:
    cov = df[df["data_year"] == (cohort_year - 1)].copy()
    pot = cov[(cov["gname"] == 0) | (cov["gname"] == cohort_year)].copy()

    # estimate cohort-specific propensity score
    lr.fit(X_ps, y_ps)
    pot["pscore"] = lr.predict_proba(X_ps)[:, 1]

    # within sic3, match on Mahalanobis distance in (log_sale, log_mv)
    dist = cdist(X_tr, X_ct, metric="mahalanobis", VI=VI)
```

This is attractive for three reasons.

### 1. It improves comparability
The design does not rely only on fixed effects to clean up large cross-sectional differences. It first tries to compare treated firms to firms that already looked similar before the event.

### 2. It respects treatment timing
Matching is done relative to each treatment cohort rather than once for the whole sample.

### 3. It makes the event-study interpretation sharper
Because the stacked panel is built around comparable treated-control sets for each cohort, the event-study plots are easier to interpret than a single fully pooled TWFE event study.

This is not a magic solution. Matching does **not** prove parallel trends. But it is a strong design enhancement, and for a recruiter or economist reading the notebook it signals that the empirical design is not naïve.


## Step D. Baseline estimation: fixed effects DiD and event studies

The core regression wrapper is intentionally simple and reusable. It runs a `PanelOLS` with:
- entity fixed effects,
- time fixed effects,
- clustered standard errors.

```python
mod = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
    check_rank=True,
)
```

For each outcome, the analysis does two related things:

### 1. Baseline DiD
Estimate the average post-treatment effect through `Post_Treated`.

### 2. Event study
Replace the single post indicator with event-time dummies interacted with treatment status, omitting `k = -1` as the reference year.

That baseline event-study machinery is important because it lets the notebook ask:
- whether effects build slowly or quickly,
- whether pre-trends are approximately flat,
- whether the treatment effect is concentrated in the deal year or after it.

The code also saves compact regression tables and coefficient plots for every outcome. That is good repo design because it makes the project reproducible and inspectable without forcing users to rerun everything.

### Caveat
The notebook should be explicit that plain two-way fixed-effects event studies can be biased under staggered timing and heterogeneous treatment effects. The project handles that appropriately by treating the baseline TWFE results as a **first screen**, not as the final word.


## Step E. Heterogeneity analysis: why the project uses `Z` interactions and triple-DiD-style specifications

A major strength of the project is that it moves beyond average effects. The analysis constructs several pre-treatment heterogeneity variables:

- baseline firm size (`Z_log_sales_cont`, plus quantile bins),
- deal size relative to market value (`Z_deal_rel_cont`, plus quantile bins),
- inventor rank within firm on cumulative patents or citations.

These are generated at the baseline period, typically `k = -1`, and then broadcast over the event window. That is the right way to do it. It keeps the heterogeneity split **predetermined** rather than allowing post-treatment variables to contaminate interpretation.

The generic event-study runner then supports a triple-DiD-style extension:

```python
df_use[post_z] = df_use["Post"] * z_num
df_use[triple] = df_use["Post_Treated"] * z_num
```

In plain language, this asks not only:

> Did treated firms change after the event?

but also:

> Did treated firms change **more or less** after the event depending on baseline size, deal scale, or inventor rank?

This is exactly the kind of heterogeneity design that makes the project look more mature. Tech economists and hiring managers are often less impressed by one average treatment effect than by a disciplined answer to **where** the effect is strongest and **why**.


## Step F. Advanced firm estimators: what each one contributes

The project then re-estimates selected outcomes with more specialized methods.

### 1. Sun–Abraham interaction-weighted event study
This is the most natural staggered-adoption robustness check for the baseline event studies. It avoids some of the weighting problems of TWFE event studies when treatment timing is staggered and treatment effects vary by cohort or over event time.

Why it matters:
- if a pattern is visible in both the matched TWFE event study and Sun–Abraham, confidence increases;
- if the shapes are very different, the TWFE result should be treated cautiously.

### 2. Borusyak–Jaravel–Spiess imputation estimator
This method imputes untreated outcomes and then constructs treatment effects from those imputed counterfactuals. It is a useful complement to Sun–Abraham because it relies on a different implementation logic.

Why it matters:
- agreement between BJS and Sun–Abraham is especially reassuring;
- disagreement is informative and should be discussed, not hidden.

### 3. Synthetic Control
The code runs firm-specific synthetic control fits and then aggregates relative gaps across successful treated units.

Why it matters:
- SCM is visually intuitive,
- it is useful for showing that some effects are not artifacts of a single regression functional form,
- and it gives a concrete unit-level interpretation.

Main caveat:
SCM is more fragile than it looks. Results depend on donor availability, pre-period fit, window choice, and whether a treated firm has enough support for a sensible synthetic counterpart.

### 4. Causal Forest
The firm analysis also runs a Causal Forest using pre-period covariates and short-run post outcomes.

This is **not** the main identification engine. It is best understood as a structured heterogeneity tool:
- which pre-period firm characteristics predict more negative or more positive treatment effects?
- does the forest agree with the simpler size-based heterogeneity splits?

That is the right way to present it. A forest can be very useful, but it should not be oversold as replacing the causal identification strategy.


## Step G. Placebo and balance checks: why they are central rather than decorative

A strong empirical notebook should make clear that credibility is built through failed tests as well as successful ones.

The analysis includes:

### Pre/post balance diagnostics
The matching design is evaluated by comparing treated and control firms on pre-period variables such as `log_sale` and `log_mv`, before and after the stacking procedure.

### Lead placebos
Treatment is artificially shifted earlier in time. If the model then finds a large effect before the actual event, that is evidence against a clean causal interpretation.

### Permutation placebos
Treatment timing is permuted across treated units. This checks whether the apparent signal is stronger than what one would expect from arbitrary timing assignments.

This is a very good robustness philosophy. It tells the reader:

- we are not just looking for significance,
- we are asking whether the effect looks specific to the actual event timing.

That stance is especially important if the project will be read by economists in tech or applied research teams, where skepticism about event-study designs is usually high.


## Step H. Inventor-year design: the mechanism side of the project

The inventor-year branch is where the project becomes especially distinctive. It asks a different question from the firm panel:

> What happens to inventors attached to treated firms, relative to matched control inventors, around merger events?

This is the more mechanism-oriented design. The firm panel tells us whether innovation or inventor flows changed at the organization level. The inventor panel tells us whether those firm-level changes are associated with:
- inventor mobility,
- changes in exploration versus exploitation,
- and changes in individual inventive output.

The workflow cycles over:
- role = `acquiror` or `target`,
- whether lagged firm controls are included,
- whether inventor-side controls are included,
- and several heterogeneity layers.

That branching structure is not accidental. It is a way of checking whether the substantive conclusions survive moderate changes in conditioning information.


## Step I. Preparing the inventor event panel: treated inventors versus matched controls

The inventor panel is built from the balanced inventor-event-study panel created in construction. The analysis then extracts a role-specific sample such as “acquiror inventors vs. control anchors” or “target inventors vs. control anchors.”

A central preparation step is:

```python
df = df[(df["ma_deal_role"] == role_l) | (df["is_control_event"] == 1)].copy()
df["Treated"] = (df["ma_deal_role"] == role_l).astype(int)
df["event_time"] = pd.to_numeric(df["years_from_ma_deal"], errors="coerce").astype(float)
df["cohort"] = pd.to_numeric(df["closest_deal_year"], errors="coerce").astype(float)
df["Post"] = (df["event_time"] >= 0).astype(int)
df["Post_Treated"] = df["Post"] * df["Treated"]
```

The identification logic is therefore parallel to the firm panel, but the unit of analysis is now an inventor-event rather than a firm-year.

A particularly strong feature is that the code can also create **within-firm rank indicators** based on cumulative patents or citations at `t = -1`. That makes it possible to ask whether higher-ranked inventors respond differently to M&A than lower-ranked inventors within the same firm context.

That is a very appealing design feature for a job-market or recruiter-facing notebook because it shows attention to the internal composition of firms, not just the average inventor.


## Step J. Baseline inventor outcomes and why they matter

The inventor-year branch focuses on a deliberately interpretable set of outcomes:

- `total_patents`
- `cites`
- `xi_real`
- `novelty_score_group`
- `exploration_inv`
- `exploitation_inv`
- `is_move_year`

This is a good outcome list. It spans three conceptually different margins:

### 1. Mobility
`is_move_year` is a direct mechanism outcome. If M&A affects organization, incentives, or match quality, inventor moves are one of the most natural places to look.

### 2. Direction of search
`exploration_inv` and `exploitation_inv` speak to whether inventors stay in familiar technology areas or move into newer ones.

### 3. Individual output and influence
Patents, citations, and `xi_real` capture different notions of inventor productivity and impact.

This mix is more persuasive than relying only on patent counts. It shows the reader that the project is not treating innovation as a one-dimensional object.


## Step K. Inventor-year advanced methods and why the project uses them selectively

The inventor panel is much larger and more computationally demanding than the firm panel, so the advanced methods are used more selectively.

### 1. CSDID / Callaway–Sant'Anna
The code runs a compact wrapper around `ATTgt`, using role-specific cohort definitions such as:
- `cs_g_year_target`,
- `cs_g_year_acquiror`,
- `cs_g_year_all`.

This is a strong choice. Inventor exposure to treatment is genuinely staggered, and a cohort-based estimator is appropriate. The code also trims weakly identified cells and keeps the post horizon modest, which is exactly what one should do when the panel is large and some cohorts are thin.

### 2. Sun–Abraham
For outcomes that are significant in the baseline inventor specification, the code reruns Sun–Abraham as a robustness check on the inventor-event panel.

### 3. Optional BJS
The inventor workflow is set up so that BJS can also be run, but the public configuration keeps this limited because of runtime and memory considerations.

### 4. Downsampling for advanced methods
The project explicitly downsamples inventor-event units before running the heavier estimators.

That is a sensible engineering compromise, not a conceptual weakness, as long as the notebook is honest about it. The right way to present it is:

- baseline DiD/event-study uses the full prepared panel;
- advanced methods are used on a carefully downsampled but balanced subset to keep the workflow feasible.

That explanation reads as practical and credible.


## Step L. What the current results appear to say

This part should be presented carefully because I do **not** have the exported result tables in this upload. The discussion below is therefore based on the current notebook draft, the active analysis code, and the earlier project discussions rather than on a fresh read of the saved CSV outputs.

With that limitation stated explicitly, the most credible interpretation still seems to be:

### 1. The clearest evidence is on the inventor / mechanism side
The project appears strongest when it studies **mobility** and **exploration** rather than trying to make a blanket claim about universal output decline.

That is attractive intellectually and strategically:
- it is more distinctive,
- it is closer to the actual micro mechanism,
- and it is easier to defend if aggregate output measures are mixed.

### 2. Acquiror-side effects look more coherent than target-side effects
Earlier runs and notes suggest that the acquiror side often produces cleaner and more interpretable patterns than the target side.

That makes economic sense. Acquirors are where integration, reallocation, and coordination decisions are actively made.

### 3. Exploration is more useful than exploitation as a headline
Because exploitation is mechanically close to the mirror image of exploration in this setup, exploration is usually the stronger variable for presentation. It carries the mechanism more naturally and avoids redundancy.

### 4. Heterogeneity by size seems substantively informative
The size-based `Z` interactions appear to matter. The emerging story is not simply “M&A hurts innovation,” but rather that the effect depends on the firm's pre-period capacity and perhaps on deal scale relative to firm size.

That is a much better research contribution than a generic average-effect statement.


## Step M. How I would present the strongest results in the final notebook

Based on the code structure and the earlier discussions, I would recommend presenting the findings in the following hierarchy.

### Headline finding
**M&A reshapes inventive organization more clearly through inventor mobility and changes in exploratory search than through a uniform collapse in output.**

### Supporting finding 1
**The acquiror side shows the cleanest post-event changes.**

### Supporting finding 2
**Firm size moderates the post-deal response.**

### Supporting finding 3
**Aggregate patent and citation effects are more mixed and should be treated cautiously.**

That ordering is important. It makes the project sound like a serious innovation-economics paper rather than an overclaimed “M&A kills innovation” story. For recruiting purposes, that is an advantage: the project reads as careful, mechanism-aware, and empirically disciplined.


## Step N. What the notebook should say explicitly about limitations

A strong public notebook should make the following limits visible.

### 1. Public-firm scope
The analysis follows inventors only when they can be linked to publicly listed firms. That improves observability but means the mobility results are not the full universe of inventor moves.

### 2. Matching improves design but does not prove exogeneity
The matched stacked design is a strong improvement over a raw pooled comparison, but it does not eliminate all concerns about differential pre-trends or event selection.

### 3. Staggered-adoption methods help, but they are still design-based estimators
Sun–Abraham, BJS, and CSDID reduce known TWFE problems, but they do not make interpretation mechanical. One still has to inspect support, pre-trends, and estimator agreement.

### 4. Advanced inventor methods are run on reduced samples
This is a practical choice to keep the workflow feasible, but it means the most computationally intensive cross-checks are not literally estimated on every row of the full panel.

### 5. Output effects are less settled than mechanism effects
The project should not oversell patent counts or citations if those outcomes are less stable across methods.

Writing these caveats directly into the notebook will improve trust rather than weaken the project.


## Summary of the analysis

The analysis workflow is one of the strongest parts of the project.

It combines:

- a careful **firm-level matched stacked event-study design**,
- an **inventor-level mechanism design** built around matched control pseudo-events,
- multiple **staggered-adoption robust estimators**,
- disciplined **heterogeneity analysis** using baseline `Z` variables,
- and a good set of **placebo and balance diagnostics**.

Just as important, the code structure reflects good empirical judgment. The project does not treat all statistically significant outputs as equal. It distinguishes between:
- baseline screens,
- robustness checks,
- and results that are actually persuasive enough to headline.

For external readers, that is exactly the right signal. It shows not only coding ability, but also the applied-economics skill of deciding **which evidence should matter most**.


## Empirical design in plain language

The project uses publicly listed firms and patent-linked inventors to study what happens around merger and acquisition events.

At a high level, there are two complementary designs:

### 1. Firm-year panel
This asks whether treated firms differ from comparable untreated firms after a deal. It is useful for aggregate outcomes such as:
- patent counts,
- citations,
- novelty and exploration measures,
- inventor flows in and out of the firm.

### 2. Inventor-year / inventor-event panel
This asks whether inventors attached to treated firms behave differently after a deal relative to matched controls. This is especially useful for:
- mobility,
- exploration,
- inventor-level productivity,
- within-firm heterogeneity.

The value of the project is that it combines both levels:
the firm panel captures organizational outcomes, while the inventor panel helps interpret the mechanism.

## Identification strategy

The main estimators are:
- baseline fixed-effects DiD,
- event studies,
- Sun–Abraham interaction-weighted event studies,
- Borusyak–Jaravel–Spiess imputation,
- selected matching and stacked-panel designs.

The reason for using multiple estimators is straightforward: staggered treatment timing can make simple two-way fixed-effects event studies misleading, especially when treatment effects evolve over time or differ across cohorts.

That is why, in interpretation, this notebook gives more weight to:
- patterns that survive across multiple estimators,
- results without severe pre-trend problems,
- economically sensible magnitudes and shapes.

It gives less weight to:
- isolated significant coefficients,
- event studies with visibly unstable pre-trends,
- very large or implausible coefficients,
- patterns that reverse sign across methods without a clear explanation.

In [ ]:
# Core imports for the notebook
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# Repository root: adjust after cloning if needed
REPO_ROOT = Path(".").resolve()

# Example output folders used by the analysis script
MODEL_OUT = REPO_ROOT / "Model_outputs"
PLOTS_OUT = MODEL_OUT / "Plots"

print("Repository root:", REPO_ROOT)
print("Model outputs:", MODEL_OUT)

## Step 1. Load the key outputs

The notebook is easiest to use if the heavy analysis has already been run and the output CSVs / plots already exist.

The helper file added to the repository can build:
- summary statistics tables,
- panel diagnostics,
- robustness scorecards,
- compact headline tables for the notebook.

The next cell assumes those helper functions are available.

In [ ]:
# These imports assume the helper file from this draft has been added to the repo.
# If the file is placed elsewhere, adjust the import path.
try:
    from notebook_additions_minimal import (
        build_headline_scorecard,
        summarize_firm_panel,
        summarize_stacked_panel,
        summarize_inventor_event_panel,
        build_result_inventory,
        write_notebook_support_outputs,
    )
    HELPERS_AVAILABLE = True
except Exception as e:
    HELPERS_AVAILABLE = False
    print("Helper import failed:", e)

## Step 2. Recommended summary statistics to include

A public-facing notebook should show basic descriptive structure before jumping into causal estimates.

I recommend including four compact tables:

1. **Firm panel summary**  
   Number of firms, years, treated firms, and baseline means for headline outcomes.

2. **Stacked firm-panel summary**  
   Number of treated and control firm-event units, balance over the event window, and treated share.

3. **Inventor-event panel summary**  
   Number of inventors, inventor-event units, treated and control units, and event-window balance.

4. **Headline baseline means**  
   Means at `t = -1` for the outcomes that appear in the narrative.

This helps readers anchor effect sizes and see the scale of the analysis.

In [ ]:
# Example usage once the relevant dataframes are loaded in memory:
#
# firm_summary = summarize_firm_panel(firm_panel, outcomes=["xi_real", "arriving_inventors_count", "exploration_firm"])
# acq_stack_summary = summarize_stacked_panel(stacked_acquiror, label="Acquiror stack")
# tgt_stack_summary = summarize_stacked_panel(stacked_target, label="Target stack")
# inv_summary = summarize_inventor_event_panel(inv_es, label="Inventor event panel")
#
# display(firm_summary)
# display(acq_stack_summary)
# display(tgt_stack_summary)
# display(inv_summary)

## Step 3. How to think about the results

A good empirical reader should evaluate the results in layers rather than asking whether one coefficient is statistically significant.

### A. What is a true headline result?
A result is headline-worthy if most of the following are true:
- the sign is stable across reasonable specifications,
- event-study pre-trends are not obviously problematic,
- the magnitude is economically interpretable,
- the estimate aligns with the mechanism story,
- more robust staggered-treatment estimators do not overturn it,
- and placebos do not generate similarly strong patterns.

### B. What is a supporting result?
A result is useful but secondary if:
- it supports the mechanism,
- it fits the economic story,
- but it is weaker, noisier, or more sensitive than the main result.

### C. What should stay out of the headline?
Results should stay out of the headline if:
- they are driven by a single estimator,
- they rely on implausibly large coefficients,
- they have visibly non-flat pre-trends,
- they reverse sign across methods without a clear explanation,
- or they are only present in a downsampled advanced run and not in the full baseline panel.

This project is strongest when interpreted with that discipline.


## Proposed interpretation of the current evidence

I want to be explicit here: I do **not** have the exported regression result files in this upload, so this section reflects the current draft notebook, the active analysis code, and the project's recent discussion history rather than a fresh verification against all saved outputs.

With that caveat stated, the current best interpretation is:

### 1. The strongest story is on the acquiror side
The clearest and most coherent results appear to be concentrated on the acquiror side rather than the target side.

This is substantively attractive because it also fits the mechanism: integration, reassignment, and organizational redesign are most directly implemented by the acquiror.

### 2. Mobility and exploration are more credible than a broad output-collapse story
The clearest evidence appears to concern:
- inventor movement,
- exploration behavior,
- and composition changes in inventive activity.

That is a better and more defensible story than claiming that M&A uniformly reduces innovation across all margins.

### 3. Firm size moderates the response
The heterogeneity design suggests that some negative or disruptive effects weaken among larger firms. That is an important result because it turns the project from a simple average-effect exercise into a richer organizational story.

### 4. Exploitation should not be the centerpiece
For presentation, it is sensible to focus on **exploration** and treat exploitation as supporting context rather than a headline outcome, because exploitation is partly the mirror image of exploration in this construction.

### 5. Aggregate output results should be framed cautiously
Patent counts, citations, and related outcomes seem more mixed across methods. They should remain in the notebook, but they should be presented as less settled than the mobility and exploration results.

That narrower interpretation makes the project stronger, not weaker.


## Draft headline paragraph for the notebook

**Draft version**

This project studies how mergers and acquisitions affect firms' inventive activity and the behavior of individual inventors. Across firm-level and inventor-level panels, the most persuasive pattern is not a uniform decline in innovation output. Instead, the data point more clearly to a reorganization of inventive effort after deals, especially on the acquiror side. Post-merger periods are associated with changes in inventor mobility and stronger exploration-oriented behavior, while broad output measures such as patents and citations are less stable across estimators and often complicated by pre-trend concerns. The evidence also suggests that firm size moderates these effects, with larger firms appearing better able to absorb or offset negative baseline impacts.

## Results section structure for the final notebook

I would organize the final notebook results section exactly like this:

### Section 1. Main descriptive setup
- sample definition,
- event window,
- matching logic,
- panel sizes.

### Section 2. Headline result: acquiror inventor mobility
- baseline DiD,
- event study,
- one advanced estimator figure,
- short interpretation.

### Section 3. Headline result: exploration
- baseline DiD,
- event study,
- advanced estimator comparison,
- explain why this is more informative than exploitation.

### Section 4. Supporting result: inventor inflows / outflows
- use only if the signs are stable enough,
- keep brief.

### Section 5. Heterogeneity by firm size
- show one clean specification,
- interpret as a moderator rather than a second headline.

### Section 6. What does *not* survive cleanly
- patent counts,
- citation-based quality outcomes,
- any result with major pre-trend problems.

## Robustness philosophy

A good public notebook does not try to make every statistically significant coefficient look equally important.

Instead, it should say:
- what survived,
- what weakened,
- what looks fragile,
- and why that matters.

That makes the project look more credible, not less.

In [ ]:
# Example: build a compact result inventory from exported significance tables
#
# inventory = build_result_inventory(MODEL_OUT)
# display(inventory.head(20))
#
# The inventory is useful for quickly checking:
# - which outcomes appear significant,
# - in which role,
# - under which family of estimators,
# - and whether they belong in the headline story.

## Suggested headline scorecard

The notebook should include one compact scorecard with rows like:

- acquiror inventor mobility,
- acquiror exploration,
- arriving inventors,
- departing inventors,
- patents,
- citations,
- innovation quality.

And columns like:

- baseline DiD,
- event study shape,
- Sun–Abraham,
- BJS,
- overall judgment.

The purpose is not to automate interpretation perfectly.  
It is to force a disciplined summary.

In [ ]:
# Example scorecard construction
#
# scorecard = build_headline_scorecard(
#     baseline_csv=MODEL_OUT / "inventor_year_event_study" / "inventor_year_event_study_significance_selected.csv",
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# display(scorecard)

## Preliminary interpretation of the scorecard

The scorecard should be read conservatively:

- **Strong** means the result is stable, interpretable, and narratively central.
- **Moderate** means it is promising but should be presented with caveats.
- **Weak / fragile** means it may still be economically interesting, but it should not drive the public-facing claim.

## Draft “main findings” bullets for the notebook

### Main finding 1
**Acquiror-side inventor mobility rises after deals.**

Why this matters:
- mobility is directly linked to organizational adjustment after M&A,
- it is easier to interpret than many abstract output metrics,
- and it helps explain why output effects may be mixed.

### Main finding 2
**Exploration increases more clearly than broad output falls.**

Why this matters:
- it suggests a change in the direction of search rather than a simple collapse in inventive activity,
- it connects naturally to post-merger reorganization,
- and it is more distinctive than exploitation, which is partly its mirror image.

### Main finding 3
**Negative baseline effects weaken for larger firms.**

Why this matters:
- it introduces economically meaningful heterogeneity,
- and it gives the paper a richer contribution than just documenting an average effect.

## Draft “what we do not claim” section

This project does **not** currently support a simple claim that M&A universally depresses innovation.

That stronger claim would require:
- cleaner agreement across estimators,
- fewer pre-trend concerns,
- and more stable output effects across the patent and citation measures.

That does not weaken the project. It sharpens it.

## Additional statistics worth adding before the GitHub version is finalized

I recommend adding the following compact extras:

1. **Baseline means at `t = -1`** for the headline outcomes.  
   This lets readers interpret magnitudes.

2. **Counts of treated and control units** in each main stacked panel.  
   This helps readers trust the sample construction.

3. **Pre-trend diagnostic table** for the headline event studies.  
   Even a simple count of significant pre-period coefficients is useful.

4. **Outcome availability table** showing how much missingness exists by outcome.  
   This is especially useful in inventor-level panels.

5. **One robustness scorecard** combining baseline and advanced methods.  
   This is probably the single most valuable addition for presentation.

## Draft sequence for LinkedIn posts

### Post 1 — The question
How do mergers and acquisitions affect innovation inside firms? Most people expect a simple output story. The data suggest something subtler.

### Post 2 — The mechanism
The clearest post-merger changes appear in inventor mobility and the direction of search. The organization changes before the output numbers fully tell a story.

### Post 3 — Exploration
One of the strongest patterns is an increase in exploration-oriented behavior on the acquiror side. That suggests reallocation and search broadening after deals.

### Post 4 — Heterogeneity
Larger firms appear better able to absorb or offset some negative baseline effects. Integration capacity may matter as much as the deal itself.

### Post 5 — Methods
Why multiple DiD estimators matter in staggered-adoption settings, and why not every significant coefficient should make it into the headline.

## Suggested next empirical additions

If you want to strengthen the notebook further, the highest-value additions are:

1. a compact summary-statistics block,
2. a pre-trend diagnostic table,
3. a results scorecard that classifies outcomes as strong / moderate / fragile,
4. one or two polished headline figures.

I would **not** expand the scope much beyond that unless a new analysis clearly improves the story.

## Final draft conclusion

The project's main contribution is not simply to ask whether M&A changes patent counts. Its deeper contribution is to trace how merger events reorganize the **people** and **search behavior** behind innovation.

The construction pipeline builds unusually rich linked panels connecting patents, inventors, firms, mobility events, technological proximity, and financial characteristics. The analysis pipeline then uses those panels in a disciplined sequence: matched stacked firm-level DiD, inventor-event designs, heterogeneity by baseline characteristics, staggered-adoption robust estimators, and placebo checks.

Taken together, the most credible evidence appears to point to a story in which the consequences of M&A show up first in **inventor mobility and exploratory search**, especially on the acquiror side, while aggregate output effects are more mixed and should be interpreted with more caution. That is a narrower claim than “M&A reduces innovation,” but it is also more persuasive, more original, and more consistent with how organizational change actually unfolds.

For a public-facing notebook, that is the right ending. It presents the project as technically serious, empirically careful, and economically thoughtful.


In [ ]:
# Optional convenience cell:
# once the heavy analysis has been run and outputs exist, use the helper file
# to create notebook-ready support CSVs in one step.
#
# support = write_notebook_support_outputs(
#     output_root=MODEL_OUT,
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# support